# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question.** Which content pages should a reviewer look at first because their search visibility is likely to fall over the next 30 days, and does a learned model rank them better than a transparent rule?

**Decision supported.** Each review cycle a content reviewer has time for roughly 20 to 50 pages out of thousands. The ranked queue tells them where to look first and why (reason codes), and which kind of action fits: refresh or improve, protect, merge or prune, or monitor.

**Frame.** For content reviewers deciding which pages to review first, we build a ranked review queue from the FlyRank ML Internship warehouse (Google Search Console and Analytics daily performance plus content metadata). The queue scores the probability that a page's impressions persistently fall, relative to its client, over the next 30 days, and is judged by precision@K against a transparent rule baseline on a sealed test month. A wrong call costs reviewer time (a false alarm) or a longer-undetected decline (a miss). A plain rule is not enough because decline depends on many tangled, shifting signals, and a flag alone cannot order a very large flagged set. We claim only observed, directional, and decision-support results: no causal effect of any refresh, and nothing about how Google ranks pages.

In [1]:
K_LIST = [20, 50]  # review capacities the queue is judged at
print('Judged at K =', K_LIST)

Judged at K = [20, 50]


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release.** The full pseudonymized warehouse release of the FlyRank ML Internship dataset (hosted on Hugging Face, access-gated). No client names, domains, URLs, queries, or raw exports appear in this work.

**Tables used**
- `fact_content_daily_performance`: one row per content item per day; 78,835,655 rows from 2025-01-27 to 2026-06-30. Exact duplicate rows (6,390 groups, 0.016%) are removed with `SELECT DISTINCT`.
- `dim_content`: 519,606 content items (static attributes, creation date, status).
- `dim_clients`: client history start dates, used for eligibility.
- `fact_content_query_90d`: a single window (2026-04-02 to 2026-06-30) that overlaps the label windows, so it is not a model feature.

**Date windows.** For each month-end snapshot t, features come from t-90d to t and the label from t to t+30d. Train: t = 2025-04-30 to 2026-03-31. Validation: t = 2026-04-30. Sealed test: t = 2026-05-31 (label window June 2026). t = 2026-06-30 is used only for the final ranked list.

**Modeling universe.** Published, not-deleted pages at least 90 days old, from clients with at least 90 days of history at t, with at least 100 impressions in the 30 days before t. Pages below the floor are reported as uncovered and not labeled (about 43% of active pages in 2026-03).

**Excluded, and why.** Deleted or unpublished pages (they carried negligible traffic in 2026-03; 411,540 of 519,606 pages are kept). `content_updated_date` (82.5% of dated pages show a May to Jul 2026 update, which looks like a bulk touch). `last_optimized_date` as a feature (no training-period coverage; used only as a flag). Export-time status fields, AI-referral sessions (sparse), and GA4 metrics unless explicitly available. Product flags (health score, priority score, action type) are not in the data and are never used.

Full contract, with the query behind every number: `w03_data_contract.ipynb`.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token: ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Design constants used by every later section
FLOOR = 100          # min impressions in the 30 days before t
FEATURE_DAYS = 90
LABEL_DAYS = 30
SNAPSHOTS = {
    'train': ['2025-04-30', '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
              '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31'],
    'validation': '2026-04-30',
    'test': '2026-05-31',
    'final_list': '2026-06-30',
}

# Cheap checks that the numbers in the text above still hold (full queries: w03_data_contract.ipynb)
con.sql(f"SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(*) AS n_rows FROM {TABLES['fact_daily']}").show()
con.sql(f"SELECT is_published, is_deleted, COUNT(*) AS n FROM {TABLES['dim_content']} GROUP BY 1,2 ORDER BY n DESC").show()
con.sql(f"SELECT MIN(gsc_data_start) AS earliest, MAX(gsc_data_start) AS latest FROM {TABLES['dim_clients']}").show()

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬──────────┐
│   min_d    │   max_d    │  n_rows  │
│    date    │    date    │  int64   │
├────────────┼────────────┼──────────┤
│ 2025-01-27 │ 2026-06-30 │ 78835655 │
└────────────┴────────────┴──────────┘

┌──────────────┬────────────┬────────┐
│ is_published │ is_deleted │   n    │
│   boolean    │  boolean   │ int64  │
├──────────────┼────────────┼────────┤
│ true         │ false      │ 411540 │
│ false        │ true       │ 101559 │
│ false        │ false      │   6507 │
└──────────────┴────────────┴────────┘

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2025-01-27 │ 2026-06-02 │
└────────────┴────────────┘



## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
import pandas as pd

def build_label_base(t_date, benchmark='universe', table='label_base'):
    """One row per eligible page at snapshot t, with the label ingredients.
    benchmark='universe': client ratio computed over the same eligible pages (like for like).
    benchmark='all': client ratio computed over every page of the client (the first version)."""
    t = pd.Timestamp(t_date)
    half = LABEL_DAYS // 2
    months = sorted({(t + pd.Timedelta(days=k)).strftime('%Y-%m') for k in (-(LABEL_DAYS - 1), 0, 1, LABEL_DAYS)})
    months_sql = ', '.join(f"'{m}'" for m in months)
    bench_src = 'elig' if benchmark == 'universe' else 'page'
    con.sql(f"""
        CREATE OR REPLACE TEMP TABLE {table} AS
        WITH daily AS (
            SELECT DISTINCT report_date, client_hash_id, content_hash_id, gsc_impressions
            FROM {TABLES['fact_daily']}
            WHERE month IN ({months_sql})
              AND report_date >  DATE '{t_date}' - {LABEL_DAYS}
              AND report_date <= DATE '{t_date}' + {LABEL_DAYS}
        ),
        page AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date <= DATE '{t_date}' THEN gsc_impressions ELSE 0 END) AS last30,
                   SUM(CASE WHEN report_date >  DATE '{t_date}'
                             AND report_date <= DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f1,
                   SUM(CASE WHEN report_date >  DATE '{t_date}' + {half} THEN gsc_impressions ELSE 0 END) AS f2
            FROM daily
            GROUP BY 1, 2
        ),
        elig AS (
            SELECT p.*
            FROM page p
            JOIN {TABLES['dim_content']} d
              ON d.client_hash_id = p.client_hash_id AND d.content_hash_id = p.content_hash_id
            JOIN {TABLES['dim_clients']} k ON k.client_hash_id = p.client_hash_id
            WHERE d.is_published AND NOT d.is_deleted
              AND d.content_created_date <= DATE '{t_date}' - {FEATURE_DAYS}
              AND k.gsc_data_start <= DATE '{t_date}' - {FEATURE_DAYS}
              AND p.last30 >= {FLOOR}
        ),
        cl AS (
            SELECT client_hash_id, SUM(last30) AS c_last30, SUM(f1) AS c_f1, SUM(f2) AS c_f2
            FROM {bench_src}
            GROUP BY 1
        )
        SELECT e.client_hash_id, e.content_hash_id, e.last30, e.f1, e.f2,
               1.0 * (e.f1 + e.f2) / e.last30 AS raw,
               1.0 * (c.c_f1 + c.c_f2) / c.c_last30 AS c_ratio,
               (1.0 * (e.f1 + e.f2) / e.last30) / (1.0 * (c.c_f1 + c.c_f2) / c.c_last30) AS rel,
               (1.0 * e.f1 / e.last30) / (1.0 * c.c_f1 / c.c_last30) AS rel_h1,
               (1.0 * e.f2 / e.last30) / (1.0 * c.c_f2 / c.c_last30) AS rel_h2
        FROM elig e
        JOIN cl c ON c.client_hash_id = e.client_hash_id
        WHERE c.c_last30 > 0 AND c.c_f1 > 0 AND c.c_f2 > 0
    """)

def centering_summary(table, label):
    return con.sql(f"""
        SELECT '{label}' AS run, COUNT(*) AS n_pages,
               ROUND(quantile_cont(raw, 0.5), 3) AS median_raw_ratio,
               ROUND(quantile_cont(rel, 0.5), 3) AS median_rel,
               ROUND(AVG(CAST(rel <= 0.8 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_8,
               ROUND(AVG(CAST(rel <= 0.5 AND rel_h1 < 1 AND rel_h2 < 1 AS INTEGER)), 3) AS base_rate_cut_0_5
        FROM {table}
    """)

# D1. Same snapshot, two benchmarks: all pages of the client vs the eligible pages only
build_label_base('2026-03-31', benchmark='all', table='lb_all')
build_label_base('2026-03-31', benchmark='universe', table='lb_universe')
centering_summary('lb_all', 'Mar-31, benchmark = all pages').show()
centering_summary('lb_universe', 'Mar-31, benchmark = eligible pages').show()

# D2. Placebo: the same like-for-like definition at two other snapshots
for t_other in ['2026-02-28', '2026-01-31']:
    build_label_base(t_other, benchmark='universe', table='lb_other')
    centering_summary('lb_other', f'{t_other}, benchmark = eligible pages').show()

# D3. How big and how fast-moving are the client benchmarks? (client_ratio > 1 = client grew)
con.sql("""
    SELECT COUNT(*) AS n_clients,
           ROUND(quantile_cont(c_ratio, 0.5), 3) AS median_client_ratio,
           quantile_cont(n_pages, [0.1, 0.5, 0.9]) AS pages_per_client_q10_50_90
    FROM (SELECT client_hash_id, ANY_VALUE(c_ratio) AS c_ratio, COUNT(*) AS n_pages
          FROM lb_universe GROUP BY 1)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│              run              │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│            varchar            │  int64  │      double      │   double   │      double       │      double       │
├───────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ Mar-31, benchmark = all pages │   64809 │            0.756 │      0.772 │             0.507 │             0.229 │
└───────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                run                 │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│              varchar               │  int64  │      double 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-02-28, benchmark = eligible pages │   55525 │            1.081 │      0.963 │             0.314 │             0.102 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────────────────────┬─────────┬──────────────────┬────────────┬───────────────────┬───────────────────┐
│                  run                   │ n_pages │ median_raw_ratio │ median_rel │ base_rate_cut_0_8 │ base_rate_cut_0_5 │
│                varchar                 │  int64  │      double      │   double   │      double       │      double       │
├────────────────────────────────────────┼─────────┼──────────────────┼────────────┼───────────────────┼───────────────────┤
│ 2026-01-31, benchmark = eligible pages │   41324 │             1.26 │       1.07 │              0.25 │             0.067 │
└────────────────────────────────────────┴─────────┴──────────────────┴────────────┴───────────────────┴───────────────────┘

┌───────────┬─────────────────────┬────────────────────────────┐
│ n_clients │ median_client_ratio │ pages_per_client_q10_50_90 │
│   int64   │       double        │          double[]          │
├───────────┼─────────────────────┼───────────────────

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
